# Entrenamiento YOLOv8n-cls — Aguacatia

**Dataset:** Hass Avocado Ripening Photographic Dataset (Mendeley, CC BY 4.0)  
**Modelo:** YOLOv8n-cls (nano, clasificación de imágenes)  
**Clases:** 5 etapas de maduración  

## Instrucciones
1. Ejecutar `Entorno` → montar Google Drive
2. Subir el ZIP del dataset a Google Drive (o usar Kaggle API)
3. Ejecutar todas las celdas en orden
4. Al final: descargar `best.pt` y copiarlo a `api/model/best.pt`

In [ ]:
# ── 1. Instalar dependencias ──────────────────────────────────────────────────
!pip install -q ultralytics==8.3.0 openpyxl scikit-learn matplotlib seaborn

In [ ]:
# ── 2. Montar Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Configuración ──────────────────────────────────────────────────────────
import os

# Ruta al ZIP en Google Drive (ajustar si es necesario)
ZIP_PATH = '/content/drive/MyDrive/aguacatia/Hass Avocado Ripening Photographic Dataset.zip'

DATASET_DIR = '/content/aguacatia_dataset'
RAW_DIR     = f'{DATASET_DIR}/raw'
PROC_DIR    = f'{DATASET_DIR}/processed'

CLASSES = ['unripe', 'breaking', 'ripe_first', 'ripe_second', 'overripe']

# Parámetros de entrenamiento
EPOCHS    = 100
IMG_SIZE  = 640
BATCH     = 32
PATIENCE  = 15  # Early stopping

print('Configuración lista')

In [ ]:
# ── 4. Extraer y preparar dataset ─────────────────────────────────────────────
import zipfile
import shutil
import openpyxl
from pathlib import Path

LABEL_MAP = {
    'unripe': 'unripe', 'breaking': 'breaking',
    'ripe': 'ripe_first', 'ripe (2)': 'ripe_second',
    'ripe_1': 'ripe_first', 'ripe_2': 'ripe_second',
    'ripe first': 'ripe_first', 'ripe second': 'ripe_second',
    'overripe': 'overripe',
}

# Crear directorios
for cls in CLASSES:
    os.makedirs(f'{RAW_DIR}/{cls}', exist_ok=True)

# Extraer ZIP
print('Extrayendo ZIP...')
tmp = '/content/_tmp'
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(tmp)

# Encontrar Excel e imágenes
tmp_path = Path(tmp)
xlsx = next(tmp_path.rglob('*.xlsx'))
images_dir = tmp_path / 'Hass Avocado Ripening Photographic Dataset' / 'Avocado Ripening Dataset'
if not images_dir.exists():
    images_dir = next(d for d in tmp_path.rglob('*') if d.is_dir() and len(list(d.glob('*.jpg'))) > 100)

print(f'Excel: {xlsx.name}')
print(f'Imágenes: {len(list(images_dir.glob("*.jpg")))} JPGs')

# Leer etiquetas
wb = openpyxl.load_workbook(xlsx, read_only=True, data_only=True)
ws = wb.active
headers = [str(c.value).strip().lower() if c.value else '' for c in next(ws.iter_rows(max_row=1))]
try:
    img_col = next(i for i, h in enumerate(headers) if 'file' in h or 'image' in h or 'name' in h)
    lbl_col = next(i for i, h in enumerate(headers) if 'ripe' in h or 'stage' in h or 'label' in h or 'class' in h)
except StopIteration:
    img_col, lbl_col = 0, len(headers) - 1

labels = {}
for row in ws.iter_rows(min_row=2, values_only=True):
    if not row[img_col]: continue
    fn = str(row[img_col]).strip()
    if not fn.endswith('.jpg'): fn += '.jpg'
    lbl = str(row[lbl_col]).strip().lower() if row[lbl_col] else ''
    mapped = LABEL_MAP.get(lbl)
    if mapped: labels[fn] = mapped
wb.close()

# Copiar imágenes a carpetas por clase
found, missing = 0, 0
for fn, cls in labels.items():
    src = images_dir / fn
    if src.exists():
        shutil.copy2(src, f'{RAW_DIR}/{cls}/{fn}')
        found += 1
    else:
        missing += 1

print(f'\nImágenes organizadas: {found} | No encontradas: {missing}')
print('\nDistribución por clase:')
for cls in CLASSES:
    n = len(list(Path(f'{RAW_DIR}/{cls}').glob('*.jpg')))
    print(f'  {cls:15s}: {n:5d}')

shutil.rmtree(tmp)

In [ ]:
# ── 5. Split train / val / test (70 / 15 / 15) ───────────────────────────────
import random
from pathlib import Path

random.seed(42)

totals = {'train': 0, 'val': 0, 'test': 0}

print(f'{"Clase":15s} {"Train":>7} {"Val":>7} {"Test":>7} {"Total":>7}')
print('-' * 45)

for cls in CLASSES:
    imgs = sorted(Path(f'{RAW_DIR}/{cls}').glob('*.jpg'))
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)

    splits = {
        'train': imgs[:n_train],
        'val':   imgs[n_train:n_train + n_val],
        'test':  imgs[n_train + n_val:],
    }
    counts = {}
    for split, split_imgs in splits.items():
        dest = Path(f'{PROC_DIR}/{split}/{cls}')
        dest.mkdir(parents=True, exist_ok=True)
        for img in split_imgs:
            shutil.copy2(img, dest / img.name)
        counts[split] = len(split_imgs)
        totals[split] += len(split_imgs)

    print(f'{cls:15s} {counts["train"]:>7} {counts["val"]:>7} {counts["test"]:>7} {n:>7}')

print('-' * 45)
grand = sum(totals.values())
print(f'{"TOTAL":15s} {totals["train"]:>7} {totals["val"]:>7} {totals["test"]:>7} {grand:>7}')

In [ ]:
# ── 6. Verificar GPU disponible ───────────────────────────────────────────────
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 7. Entrenamiento YOLOv8n-cls ─────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')  # Descarga automática

results = model.train(
    data=PROC_DIR,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/aguacatia_runs',
    name='yolov8n_cls_aguacatia',
    exist_ok=True,
    # Aumentos de datos para mejorar generalización
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.1,
    fliplr=0.5,
)

print('\nEntrenamiento completado')
print('Mejor modelo:', results.save_dir)

In [ ]:
# ── 8. Evaluación en conjunto de test ────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from pathlib import Path
from PIL import Image

best_model = YOLO(f'/content/aguacatia_runs/yolov8n_cls_aguacatia/weights/best.pt')

y_true, y_pred, y_proba = [], [], []

for cls_idx, cls in enumerate(CLASSES):
    test_dir = Path(f'{PROC_DIR}/test/{cls}')
    for img_path in test_dir.glob('*.jpg'):
        result = best_model.predict(str(img_path), verbose=False)[0]
        probs = result.probs.data.cpu().numpy()
        pred_idx = int(np.argmax(probs))
        y_true.append(cls_idx)
        y_pred.append(pred_idx)
        y_proba.append(probs)

y_true  = np.array(y_true)
y_pred  = np.array(y_pred)
y_proba = np.array(y_proba)

# Reporte de clasificación
print('=== Reporte de clasificación ===')
print(classification_report(y_true, y_pred, target_names=CLASSES))

# Accuracy
accuracy = (y_true == y_pred).mean()
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')

In [ ]:
# ── 9. Matriz de confusión ────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Matriz de Confusión (valores absolutos)')
axes[0].set_ylabel('Etiqueta real')
axes[0].set_xlabel('Predicción del modelo')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Matriz de Confusión (normalizada)')
axes[1].set_ylabel('Etiqueta real')
axes[1].set_xlabel('Predicción del modelo')

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: /content/confusion_matrix.png')

In [ ]:
# ── 10. Curvas ROC y AUC ──────────────────────────────────────────────────────
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(y_true, classes=list(range(len(CLASSES))))

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#22c55e', '#eab308', '#f97316', '#a855f7', '#ef4444']

auc_scores = []
for i, (cls, color) in enumerate(zip(CLASSES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
    auc = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
    auc_scores.append(auc)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title(f'Curvas ROC por clase | AUC macro = {np.mean(auc_scores):.3f}')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nAUC por clase:')
for cls, auc in zip(CLASSES, auc_scores):
    print(f'  {cls:15s}: {auc:.4f}')
print(f'AUC macro: {np.mean(auc_scores):.4f}')

In [ ]:
# ── 11. Guardar modelo en Google Drive ───────────────────────────────────────
import shutil

drive_dest = '/content/drive/MyDrive/aguacatia/'
os.makedirs(drive_dest, exist_ok=True)

shutil.copy(
    '/content/aguacatia_runs/yolov8n_cls_aguacatia/weights/best.pt',
    f'{drive_dest}/best.pt'
)
shutil.copy('/content/confusion_matrix.png', f'{drive_dest}/confusion_matrix.png')
shutil.copy('/content/roc_curves.png', f'{drive_dest}/roc_curves.png')

print('Archivos guardados en Google Drive:')
print(f'  {drive_dest}best.pt')
print(f'  {drive_dest}confusion_matrix.png')
print(f'  {drive_dest}roc_curves.png')
print()
print('SIGUIENTE PASO:')
print('  1. Descargar best.pt de Google Drive')
print('  2. Copiarlo a api/model/best.pt en el repositorio')
print('  3. Hacer deploy en Coolify')